In [ ]:
#Import needed libraries

import yfinance as yf
from ta.momentum import RSIIndicator
from ta.trend import MACD
import matplotlib.pyplot as plt
from xgboost import XGBClassifier
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

In [ ]:
#Using Nvidia's stock data

nvda = yf.download("NVDA", start="2020-01-01", end="2020-12-31")
nvda.to_csv("nvda_data.csv")

In [ ]:
#Implementing features  

rsi = RSIIndicator(close = nvda["Close"]["NVDA"], window=14, fillna=False)
nvda["rsi"] = rsi.rsi()
nvda["macd"] = MACD(nvda["Close"]["NVDA"]).macd_diff()
nvda["returns"] = nvda["Close"]["NVDA"].pct_change()
nvda["ma_10"] = nvda["Close"]["NVDA"].rolling(10).mean()

In [ ]:
#Check if the next day's closing price went up 

nvda["Target"] = (nvda["Close"]["NVDA"].shift(-1) > nvda["Close"]["NVDA"]).astype(int)

In [ ]:
#Extracting data needed for training the model 

features = nvda[["rsi","macd","returns","ma_10"]].dropna()
labels = nvda["Target"].loc[features.index]

x_train, x_test, y_train, y_test = train_test_split(features, labels, test_size=0.2, shuffle=False)

In [ ]:
#Initialize model and predict

from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
x_train_scaled = scaler.fit_transform(x_train)
x_test_scaled = scaler.transform(x_test)

model = LogisticRegression(class_weight='balanced')
model.fit(x_train_scaled, y_train)
y_pred = model.predict(x_test_scaled)

#print(y_train.value_counts())
#print(y_test.value_counts())
print(classification_report(y_test, y_pred))

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = RandomForestClassifier(class_weight='balanced', random_state=42)
rf_model.fit(x_train, y_train)
y_pred = rf_model.predict(x_test)
print(classification_report(y_test, y_pred))

In [ ]:
xgb_model = XGBClassifier()
xgb_model.fit(x_train, y_train)
y_pred = xgb_model.predict(x_test)

print(classification_report(y_test, y_pred))